# FrameFast on Colab

Runs the full FrameFast optimization ladder for **CogVideoX-2B** on the Colab GPU and renders
`docs/results.md`. Use *Runtime → Change runtime type → GPU* first. An A100/H100 runtime is
recommended (FP8 rungs need compute capability ≥ 8.9, e.g. L4/H100). On a 16 GB T4 the fp16 baseline
may not fit at 49 frames × 480×720; the harness records an OOM per variant and moves on, and the
`cpu_offload_*` / `vae_tiling` rungs show what does fit.

Nothing in this repository contains measured numbers until you run this notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

In [ ]:
REPO = "https://github.com/WenxueWu0216/FrameFast.git"
BRANCH = "claude/framefast-inference-optimization-n0vu0b"  # or main once merged
!git clone -b {BRANCH} {REPO} FrameFast || (cd FrameFast && git pull)
%cd FrameFast

## Choose a config

* `configs/gpu_quick.toml` — a few minutes; short ladder, 5 steps (writes `runs/gpu_quick/results.md`).
* `configs/gpu_default.toml` — full ladder, 20 steps, 8 timed runs (writes `docs/results.md`).
* `configs/gpu_full.toml` — 50 steps, 10 runs, 3 prompts × 2 seeds.

Set `TRT = "1"` to also install torch-tensorrt for the TensorRT rung (it pins a torch version and may reinstall torch).

In [ ]:
CONFIG = "configs/gpu_quick.toml"
TRT = "0"
!CONFIG={CONFIG} FRAMEFAST_TRT={TRT} bash scripts/run_on_gpu.sh

In [ ]:
from pathlib import Path

import tomllib
from IPython.display import Markdown, display

cfg = tomllib.loads(Path(CONFIG).read_text())
display(Markdown(Path(cfg.get("results_md", "docs/results.md")).read_text()))
display(Markdown(Path("docs/profile_top_ops.md").read_text()))

## Keep the results

Download `docs/results.md`, `docs/profile_top_ops.md` and `runs/*/results.json`, or commit them from here:

In [ ]:
# !git config user.email you@example.com && git config user.name "Your Name"
# !git add docs/results.md docs/profile_top_ops.md && git commit -m "Add GPU results" && git push